In [ ]:
import sys
import os

# Add the src directory to the system path to allow importing custom modules
sys.path.insert(0, os.path.abspath('../src'))

# Enable autoreload to automatically reload modules when they are edited
%load_ext autoreload
%autoreload 2


# Fetch and prepare data for prediction

* identify time frame for future feature time frame 
* read and prepare data from SMARD
* fetch and prepare weather data from open-meteo API
* combine and create features for energy and weather data -> future features
* use best model and future features for prediction

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from fetch_prepare_data import *
from train_model_predict import *

chosen_date = '2026-05-15'  
df_future = prepare_data_for_next_day_prediction(prediction_date=chosen_date)

if df_future.empty:
    print("No future features prepared. Check the prepare_data_for_next_day_prediction function.")
else:
    best_model_lgbm = load_model_from_pickle('../models/best_lgbm_model_bayesian.pkl')
    best_model_rf   = load_model_from_pickle('../models/best_rf_model_bayesian.pkl')
    best_model_xgb  = load_model_from_pickle('../models/best_xgb_model_bayesian.pkl')

    X_future = df_future.drop(columns=['time', 'EnergyDemand'], errors='ignore')

    # reindex to model's training feature order to avoid positional mismatch
    X_lgbm = X_future.reindex(columns=best_model_lgbm.feature_names_in_)
    X_rf   = X_future.reindex(columns=best_model_rf.feature_names_in_)
    X_xgb  = X_future.reindex(columns=best_model_xgb.feature_names_in_)

    df_pred_lgbm = best_model_lgbm.predict(X_lgbm)
    df_pred_rf   = best_model_rf.predict(X_rf)
    df_pred_xgb  = best_model_xgb.predict(X_xgb)

    # plot all predictions
    plt.figure(figsize=(10, 4))
    plt.plot(df_future['time'], df_pred_lgbm, label='LGBM Predictions')
    plt.plot(df_future['time'], df_pred_rf,   label='RF Predictions')
    plt.plot(df_future['time'], df_pred_xgb,  label='XGBoost Predictions')
    plt.xlabel('Time')
    plt.ylabel('Energy Demand')
    plt.title('Future Energy Demand Predictions')
    plt.legend()
    plt.show()


---
# Setup: Imports and Model Loading

* Set European locale
* Load the best models - LightGBM, Random Forest and XGBoost

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

from datetime import date, timedelta, datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from fetch_prepare_data import *
from train_model_predict import *

# ── Set European locale so date-picker calendars start on Monday ──────────────
display(HTML('<script>document.documentElement.lang = "de-DE";</script>'))

# ── Load both models once at kernel start ─────────────────────────────────────
_MODEL_PATHS = {
    'LGBM':          '../models/best_lgbm_model_bayesian.pkl',
    'Random Forest': '../models/best_rf_model_bayesian.pkl',
    'XGBoost':       '../models/best_xgb_model_bayesian.pkl'
}
models = {name: load_model_from_pickle(path) for name, path in _MODEL_PATHS.items()}
print("Models loaded:", list(models.keys()))


---
# Part 1 — Future Prediction

- **Predict for Tomorrow** — forecast the full next day (00:00–23:00 UTC)  

Results are shown as a line plot **and** a table side-by-side.  
Calendar week starts on **Monday** (European style via `lang="de-DE"`).


In [ ]:
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

# SMARD filter ID for official consumption forecast (Prognostizierter Stromverbrauch: Netzlast)
FILTER_SMARD_FORECAST = 411

# ── Widgets ───────────────────────────────────────────────────────────────────
future_model_dd = widgets.Dropdown(
    options=list(models.keys()),
    value='LGBM',
    description='Model:',
    layout=widgets.Layout(width='220px'),
)

now_label = widgets.HTML(
    value=f'<b>Now (UTC):</b> {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")}',
)

btn_tomorrow = widgets.Button(
    description='Predict for Tomorrow',
    button_style='primary',
    icon='arrow-right',
    layout=widgets.Layout(width='210px'),
)

status_future = widgets.Label(value='Ready.')
output_future = widgets.Output()


# ── Helper: run prediction and render plot + table ────────────────────────────
def _render_future(df_future, predictions, model_name, title,
                   x_fmt='%H:%M', x_label='Hour (UTC)', df_smard_fc=None):
    fig, (ax_plot, ax_tbl) = plt.subplots(
        1, 2, figsize=(14, 5),
        gridspec_kw={'width_ratios': [2.5, 1]},
    )
    if df_smard_fc is not None and not df_smard_fc.empty:
        ax_plot.plot(df_smard_fc['time'], df_smard_fc['EnergyDemand'],
                     color='mediumseagreen', linewidth=1.5, linestyle='-.',
                     label='SMARD official forecast')
    ax_plot.plot(df_future['time'], predictions, linewidth=2, color='darkorange', linestyle='--',
                 label=f'{model_name} prediction')
    ax_plot.xaxis.set_major_formatter(mdates.DateFormatter(x_fmt))
    ax_plot.set_xlabel(x_label)
    ax_plot.set_ylabel('Energy Demand (MWh)')
    ax_plot.set_title(title)
    ax_plot.legend()
    ax_plot.grid(True, alpha=0.3)
    fig.autofmt_xdate()

    df_tbl = df_future[['time']].copy()
    df_tbl['predicted_MWh'] = predictions.round(0).astype(int)
    df_tbl['Date / Hour (UTC)'] = df_tbl['time'].dt.strftime('%m-%d %H:%M')
    df_tbl = df_tbl[['Date / Hour (UTC)', 'predicted_MWh']].reset_index(drop=True)

    ax_tbl.axis('off')
    tbl = ax_tbl.table(
        cellText=df_tbl.values,
        colLabels=df_tbl.columns,
        loc='center',
        cellLoc='right',
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.1)

    plt.tight_layout()
    plt.show()


# ── Button handler ────────────────────────────────────────────────────────────
def on_tomorrow_clicked(_b):
    now_label.value = f'<b>Now (UTC):</b> {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")}'
    tomorrow_str  = (date.today() + timedelta(days=1)).isoformat()
    model_name    = future_model_dd.value

    with output_future:
        clear_output(wait=True)

        # 1. Fetch SMARD official consumption forecast for tomorrow ────────────
        status_future.value = 'Fetching SMARD official forecast for tomorrow …'
        try:
            df_smard_fc = fetch_smard_netzlast(tomorrow_str, tomorrow_str,
                                               filter_id=FILTER_SMARD_FORECAST)
        except Exception:
            df_smard_fc = pd.DataFrame(columns=['time', 'EnergyDemand'])

        # 2. Prepare ML features and predict ──────────────────────────────────
        status_future.value = f'Preparing features for {tomorrow_str} …'
        try:
            df_future = prepare_data_for_next_day_prediction(prediction_date=tomorrow_str)
        except Exception as exc:
            status_future.value = f'Feature preparation failed: {exc}'
            return
        if df_future.empty:
            status_future.value = 'No features returned — check API connectivity.'
            return

        X     = df_future.drop(columns=['time', 'EnergyDemand'], errors='ignore')
        model = models[model_name]
        if hasattr(model, 'feature_names_in_'):
            X = X.reindex(columns=model.feature_names_in_)
        preds = model.predict(X)

        _render_future(
            df_future, preds, model_name,
            title=f'Energy Demand Forecast — Tomorrow ({tomorrow_str})  [{model_name}]',
            df_smard_fc=df_smard_fc if not df_smard_fc.empty else None,
        )
        status_future.value = f'Done — {tomorrow_str} ({model_name})'


btn_tomorrow.on_click(on_tomorrow_clicked)

# ── Layout ────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    now_label,
    widgets.HBox(
        [future_model_dd, btn_tomorrow, status_future],
        layout=widgets.Layout(align_items='center', gap='12px'),
    ),
    output_future,
]))

---
# Part 2 — Historical Comparison: Actual vs SMARD Forecast vs ML Prediction

Select a **From** and **To** date to compare three demand series side by side:
- **Actual** — SMARD realized consumption (filter 410)
- **SMARD official forecast** — SMARD day-ahead consumption forecast (filter 411)
- **ML Prediction** — model prediction from features & weather data

- Maximum range: **1 year** — a warning is shown in the GUI if exceeded  

- Calendar week starts on **Monday** (European style)  
- MAE and RMSE vs actual are reported in a table below the chart

In [ ]:
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

MAX_RANGE_DAYS = 365

# SMARD filter ID for official consumption forecast (Prognostizierter Stromverbrauch: Netzlast)
FILTER_SMARD_FORECAST = 411

# ── Default range: last 7 days (SMARD data available up to yesterday) ─────────
_default_to   = date.today() - timedelta(days=1)
_default_from = _default_to - timedelta(days=6)

hist_model_dd = widgets.Dropdown(
    options=list(models.keys()),
    value='LGBM',
    description='Model:',
    layout=widgets.Layout(width='220px'),
)

date_from = widgets.DatePicker(
    description='From:',
    value=_default_from,
    min=date(2019, 1, 1),
    max=date.today() - timedelta(days=1),
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px'),
)

date_to = widgets.DatePicker(
    description='To:',
    value=_default_to,
    min=date(2019, 1, 1),
    max=date.today() - timedelta(days=1),
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px'),
)

range_warning = widgets.HTML(value='')

btn_compare = widgets.Button(
    description='Compare Prediction vs Actual',
    button_style='warning',
    icon='bar-chart',
    layout=widgets.Layout(width='270px'),
)

status_hist  = widgets.Label(value='Ready.')
output_hist  = widgets.Output()


# ── Live range validation ─────────────────────────────────────────────────────
def _validate_range(_change=None):
    try:
        delta = (date_to.value - date_from.value).days
        if delta < 0:
            range_warning.value = (
                '<span style="color:red; font-weight:bold;">'
                '⚠ "To" date must be on or after "From" date.</span>'
            )
            btn_compare.disabled = True
        elif delta > MAX_RANGE_DAYS:
            range_warning.value = (
                f'<span style="color:orange; font-weight:bold;">'
                f'⚠ Selected range is {delta} days — maximum allowed is {MAX_RANGE_DAYS} days (1 year). '
                f'Please narrow the selection.</span>'
            )
            btn_compare.disabled = True
        else:
            range_warning.value = (
                f'<span style="color:green;">Range: {delta + 1} day(s)  ✓</span>'
            )
            btn_compare.disabled = False
    except Exception:
        range_warning.value = ''


date_from.observe(_validate_range, names='value')
date_to.observe(_validate_range, names='value')
_validate_range()  # initialise on cell run


# ── Compare button handler ────────────────────────────────────────────────────
def on_compare_clicked(_b):
    from_str   = str(date_from.value)
    to_str     = str(date_to.value)
    model_name = hist_model_dd.value

    with output_hist:
        clear_output(wait=True)

        # 1. Fetch actual SMARD realized data (filter 410) ────────────────────
        status_hist.value = f'Fetching actual SMARD data for {from_str} → {to_str} …'
        try:
            df_actual = fetch_smard_netzlast(from_str, to_str)
        except Exception as exc:
            status_hist.value = f'SMARD fetch failed: {exc}'
            return
        if df_actual.empty:
            status_hist.value = 'No SMARD data available for the selected range.'
            return

        # 2. Fetch SMARD official consumption forecast (filter 411) ───────────
        status_hist.value = 'Fetching SMARD official consumption forecast …'
        try:
            df_smard_fc = fetch_smard_netzlast(from_str, to_str, filter_id=FILTER_SMARD_FORECAST)
        except Exception as exc:
            df_smard_fc = pd.DataFrame(columns=['time', 'EnergyDemand'])
            status_hist.value = f'SMARD forecast fetch failed (non-fatal): {exc}'

        # 3. Build feature matrix for the same range ──────────────────────────
        status_hist.value = 'Building model features (fetching energy + weather history) …'
        HISTORY_DAYS = 15
        try:
            hist_start = (
                pd.to_datetime(from_str) - pd.Timedelta(days=HISTORY_DAYS)
            ).strftime('%Y-%m-%d')

            df_energy = fetch_smard_netzlast(hist_start, to_str)
            df_energy = create_energy_features(df_energy)
            df_energy = create_time_based_features(
                df_energy, in_year=pd.to_datetime(to_str).year
            )

            df_weather = prepare_weather_data(in_start_date=hist_start, in_end_date=to_str)
            df_feat    = combine_energy_weather_dataset(df_energy, df_weather)
            df_feat    = df_feat.sort_values('time').reset_index(drop=True)

            # Clip to the requested range
            from_ts = pd.to_datetime(from_str, utc=True)
            to_ts   = pd.to_datetime(to_str,   utc=True) + pd.Timedelta(hours=23)
            df_feat = df_feat[
                (df_feat['time'] >= from_ts) & (df_feat['time'] <= to_ts)
            ].reset_index(drop=True)

        except Exception as exc:
            status_hist.value = f'Feature preparation failed: {exc}'
            return

        if df_feat.empty:
            status_hist.value = 'No feature data for the selected range.'
            return

        # 4. Predict ──────────────────────────────────────────────────────────
        status_hist.value = f'Running {model_name} …'
        X     = df_feat.drop(columns=['time', 'EnergyDemand'], errors='ignore')
        model = models[model_name]
        if hasattr(model, 'feature_names_in_'):
            X = X.reindex(columns=model.feature_names_in_)
        preds      = model.predict(X)
        times_pred = df_feat['time']

        # 5. Align all three series on shared timestamps ──────────────────────
        s_pred   = pd.Series(preds,   index=times_pred,              name='ML Prediction')
        s_actual = df_actual.set_index('time')['EnergyDemand'].rename('Actual')
        s_smard  = (
            df_smard_fc.set_index('time')['EnergyDemand'].rename('SMARD Forecast')
            if not df_smard_fc.empty else pd.Series(dtype=float, name='SMARD Forecast')
        )
        df_plot = pd.concat([s_actual, s_smard, s_pred], axis=1)

        # 6. Plot ─────────────────────────────────────────────────────────────
        days_in_range = (date_to.value - date_from.value).days + 1
        if days_in_range <= 3:
            x_fmt = '%m-%d %H:%M'
        elif days_in_range <= 31:
            x_fmt = '%Y-%m-%d'
        else:
            x_fmt = '%Y-%m'

        fig, ax = plt.subplots(figsize=(14, 5))
        ax.plot(df_plot.index, df_plot['Actual'],
                color='steelblue', linewidth=1.5, label='Actual (SMARD realized)')
        if not df_smard_fc.empty and df_plot['SMARD Forecast'].notna().any():
            ax.plot(df_plot.index, df_plot['SMARD Forecast'],
                    color='mediumseagreen', linewidth=1.5, linestyle='-.',
                    label='SMARD official forecast')
        ax.plot(df_plot.index, df_plot['ML Prediction'],
                color='darkorange', linewidth=1.5, linestyle='--',
                label=f'ML Prediction ({model_name})')
        ax.xaxis.set_major_formatter(mdates.DateFormatter(x_fmt))
        ax.set_xlabel('Date / Time (UTC)')
        ax.set_ylabel('Energy Demand (MWh)')
        ax.set_title(
            f'Energy Demand — Actual vs SMARD Forecast vs ML Prediction\n'
            f'{from_str} to {to_str}  [{model_name}]'
        )
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()

        # 7. Metrics table ────────────────────────────────────────────────────
        df_ml_cmp = df_plot[['Actual', 'ML Prediction']].dropna()
        mae_ml    = (df_ml_cmp['Actual'] - df_ml_cmp['ML Prediction']).abs().mean()
        rmse_ml   = ((df_ml_cmp['Actual'] - df_ml_cmp['ML Prediction']) ** 2).mean() ** 0.5

        metrics_html = (
            '<table style="border-collapse:collapse; font-family:monospace; font-size:13px;">'
            '<tr style="background:#f0f0f0;">'
            '<th style="padding:4px 16px; border:1px solid #ccc;">Series</th>'
            '<th style="padding:4px 16px; border:1px solid #ccc;">MAE (MWh)</th>'
            '<th style="padding:4px 16px; border:1px solid #ccc;">RMSE (MWh)</th>'
            '<th style="padding:4px 16px; border:1px solid #ccc;">Points</th></tr>'
            f'<tr><td style="padding:4px 16px; border:1px solid #ccc;">ML Prediction ({model_name})</td>'
            f'<td style="padding:4px 16px; border:1px solid #ccc; text-align:right;">{mae_ml:,.0f}</td>'
            f'<td style="padding:4px 16px; border:1px solid #ccc; text-align:right;">{rmse_ml:,.0f}</td>'
            f'<td style="padding:4px 16px; border:1px solid #ccc; text-align:right;">{len(df_ml_cmp)}</td></tr>'
        )
        if not df_smard_fc.empty:
            df_smard_cmp = df_plot[['Actual', 'SMARD Forecast']].dropna()
            mae_smard    = (df_smard_cmp['Actual'] - df_smard_cmp['SMARD Forecast']).abs().mean()
            rmse_smard   = ((df_smard_cmp['Actual'] - df_smard_cmp['SMARD Forecast']) ** 2).mean() ** 0.5
            metrics_html += (
                '<tr><td style="padding:4px 16px; border:1px solid #ccc;">SMARD official forecast</td>'
                f'<td style="padding:4px 16px; border:1px solid #ccc; text-align:right;">{mae_smard:,.0f}</td>'
                f'<td style="padding:4px 16px; border:1px solid #ccc; text-align:right;">{rmse_smard:,.0f}</td>'
                f'<td style="padding:4px 16px; border:1px solid #ccc; text-align:right;">{len(df_smard_cmp)}</td></tr>'
            )
        metrics_html += '</table>'
        display(widgets.HTML(metrics_html))

        status_hist.value = f'Done — {from_str} → {to_str} ({model_name})'


btn_compare.on_click(on_compare_clicked)

# ── Layout ────────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HBox(
        [date_from, date_to, hist_model_dd],
        layout=widgets.Layout(align_items='flex-end', gap='12px'),
    ),
    widgets.HBox(
        [btn_compare, status_hist],
        layout=widgets.Layout(align_items='center', gap='12px'),
    ),
    range_warning,
    output_hist,
]))